# Parkinson's Disease Detection using Deep Learning
Dataset: Oxford Parkinson's Dataset (UCI Machine Learning Repository)
Model: PyTorch Neural Network

### Step 1: Import libraries

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# check pytorch version
print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.0


### Step 2: Load the dataset

In [5]:
df = pd.read_csv("data/raw/parkinsons.data")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (195, 24)


,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


### Step 3: Check missing values and target distribution

In [6]:
# check for missing values
print("Missing values:")
print(df.isnull().sum())

# check target counts (0 = healthy, 1 = parkinsons)
print("\nTarget counts:")
print(df['status'].value_counts())

Missing values:
name                0
MDVP:Fo(Hz)         0
MDVP:Fhi(Hz)        0
MDVP:Flo(Hz)        0
MDVP:Jitter(%)      0
MDVP:Jitter(Abs)    0
MDVP:RAP            0
MDVP:PPQ            0
Jitter:DDP          0
MDVP:Shimmer        0
MDVP:Shimmer(dB)    0
Shimmer:APQ3        0
Shimmer:APQ5        0
MDVP:APQ            0
Shimmer:DDA         0
NHR                 0
HNR                 0
status              0
RPDE                0
DFA                 0
spread1             0
spread2             0
D2                  0
PPE                 0
dtype: int64

Target counts:
status
1    147
0     48
Name: count, dtype: int64


### Step 4: Split into train and test sets

In [7]:
# drop name column and separate features (X) and label (y)
X = df.drop(columns=['name', 'status']).values
y = df['status'].values

# 80% train, 20% test with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 156
Testing samples: 39


### Step 5: Feature scaling and convert to PyTorch tensors

In [8]:
# fit scaler only on training data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# create dataloader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
print("Data ready for training.")

Data ready for training.


### Step 6: Define Neural Network model

In [9]:
# 4-layer MLP model
class ParkinsonModel(nn.Module):
    def __init__(self, input_dim):
        super(ParkinsonModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.act1 = nn.LeakyReLU(0.1)
        self.drop1 = nn.Dropout(0.3)
        
        self.fc2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.act2 = nn.LeakyReLU(0.1)
        self.drop2 = nn.Dropout(0.2)
        
        self.fc3 = nn.Linear(32, 16)
        self.act3 = nn.LeakyReLU(0.1)
        
        self.out = nn.Linear(16, 1)

    def forward(self, x):
        x = self.drop1(self.act1(self.bn1(self.fc1(x))))
        x = self.drop2(self.act2(self.bn2(self.fc2(x))))
        x = self.act3(self.fc3(x))
        x = self.out(x)
        return x

model = ParkinsonModel(input_dim=22)
print(model)

ParkinsonModel(
  (fc1): Linear(in_features=22, out_features=64, bias=True)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): LeakyReLU(negative_slope=0.1)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act2): LeakyReLU(negative_slope=0.1)
  (drop2): Dropout(p=0.2, inplace=False)
  (fc3): Linear(in_features=32, out_features=16, bias=True)
  (act3): LeakyReLU(negative_slope=0.1)
  (out): Linear(in_features=16, out_features=1, bias=True)
)


### Step 7: Train the model

In [10]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)

epochs = 120

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)

    total_loss = epoch_loss / len(train_loader.dataset)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss:.4f}")

Epoch [20/120] - Loss: 0.2411
Epoch [40/120] - Loss: 0.1319
Epoch [60/120] - Loss: 0.1536
Epoch [80/120] - Loss: 0.1365
Epoch [100/120] - Loss: 0.1407
Epoch [120/120] - Loss: 0.0804


### Step 8: Evaluate model on test data

In [11]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    test_probs = torch.sigmoid(test_outputs).numpy()
    preds = (test_probs >= 0.5).astype(int)

y_test_numpy = y_test.numpy().flatten()
preds_numpy = preds.flatten()

print("Accuracy :", round(accuracy_score(y_test_numpy, preds_numpy), 4))
print("Precision:", round(precision_score(y_test_numpy, preds_numpy), 4))
print("Recall   :", round(recall_score(y_test_numpy, preds_numpy), 4))
print("F1 Score :", round(f1_score(y_test_numpy, preds_numpy), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_numpy, preds_numpy))

print("\nClassification Report:")
print(classification_report(y_test_numpy, preds_numpy, target_names=["Healthy (0)", "Parkinsons (1)"]))

Accuracy : 0.9744
Precision: 1.0
Recall   : 0.9655
F1 Score : 0.9825

Confusion Matrix:
[[10  0]
 [ 1 28]]

Classification Report:
                precision    recall  f1-score   support

   Healthy (0)       0.91      1.00      0.95        10
Parkinsons (1)       1.00      0.97      0.98        29

      accuracy                           0.97        39
     macro avg       0.95      0.98      0.97        39
  weighted avg       0.98      0.97      0.97        39



### Step 9: Test on a sample patient record

In [12]:
# test sample
sample = X_test[0].unsqueeze(0)
actual = int(y_test[0].item())

model.eval()
with torch.no_grad():
    prob = torch.sigmoid(model(sample)).item()

pred = 1 if prob >= 0.5 else 0

print("Actual Label   :", actual, "(Healthy)" if actual == 0 else "(Parkinson's)")
print("Predicted Label:", pred, "(Healthy)" if pred == 0 else "(Parkinson's)")
print(f"Confidence     : {prob if pred == 1 else 1 - prob:.2%}")

Actual Label   : 0 (Healthy)
Predicted Label: 0 (Healthy)
Confidence     : 83.63%
